
##  Categorical Data Analysis & Target Variable Analysis



### Objective
This notebook analyzes categorical features, category frequencies, rare/high-cardinality categories, class imbalance, and the target variable. It includes reusable code and a small example dataset so it can run immediately. Replace the example dataset with your own dataset when ready.



## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")

## 2. Load the Dataset

The example dataset below is designed to demonstrate both categorical analysis and a classification target. To use your own file, replace the example creation with `pd.read_csv("your_file.csv")` or the appropriate reader.

In [ ]:
# Example dataset (synthetic)
df = pd.DataFrame({
    "Age": [22, 25, 31, 45, 28, 36, 52, 41, 29, 33, 26, 48, 39, 24, 57, 30],
    "Department": ["Sales", "IT", "IT", "HR", "Sales", "Finance", "IT", "HR",
                   "Sales", "Finance", "IT", "HR", "Sales", "Finance", "IT", "Sales"],
    "Education": ["UG", "PG", "PG", "UG", "UG", "PG", "PG", "UG",
                  "UG", "PG", "UG", "PG", "PG", "UG", "PG", "UG"],
    "City": ["Chennai", "Salem", "Chennai", "Madurai", "Salem", "Chennai",
             "Coimbatore", "Chennai", "Salem", "Madurai", "Chennai", "Salem",
             "Coimbatore", "Chennai", "Madurai", "Salem"],
    "Purchased": ["No", "Yes", "Yes", "No", "No", "Yes", "Yes", "No",
                  "No", "Yes", "Yes", "No", "Yes", "No", "Yes", "No"]
})

# For your own CSV dataset, use:
# df = pd.read_csv("your_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

## 3. Configure Analysis

Set `TARGET_COLUMN` to the column you want to predict. Set `PROBLEM_TYPE` to `"classification"` or `"regression"`. If your dataset has no target column, set `TARGET_COLUMN = None`.

In [ ]:
TARGET_COLUMN = "Purchased"   # Change this to your target column, or None
PROBLEM_TYPE = "classification"  # Choose: "classification" or "regression"

if TARGET_COLUMN is not None and TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' was not found in the dataset.")

print("Target column:", TARGET_COLUMN)
print("Problem type:", PROBLEM_TYPE)
print("Rows:", len(df), "| Columns:", len(df.columns))

#  Categorical Data Analysis

## 4. Identify Categorical Columns

Categorical columns represent groups or labels, such as department, city, product type, or customer segment. This notebook treats text/object and pandas categorical columns as categorical. You can add other columns manually if numeric values are actually category codes.

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Exclude target from feature-only categorical analysis; target is analyzed separately later.
categorical_features = [
    col for col in categorical_columns if col != TARGET_COLUMN
]

print("Categorical columns:", categorical_columns)
print("Categorical feature columns:", categorical_features)

## 5. Unique Categories and Cardinality

**Cardinality** is the number of distinct values in a column. Low-cardinality columns have relatively few categories; high-cardinality columns have many. Thresholds depend on the dataset and business context.

In [ ]:
cardinality_rows = []

for col in categorical_features:
    unique_count = df[col].nunique(dropna=False)
    if unique_count <= 10:
        cardinality = "Low"
    else:
        cardinality = "High"
    cardinality_rows.append({
        "Column": col,
        "Unique Categories": unique_count,
        "Cardinality": cardinality,
        "Missing Values": int(df[col].isna().sum())
    })

cardinality_df = pd.DataFrame(cardinality_rows)
display(cardinality_df)

for col in categorical_features:
    print(f"\n{col} — unique categories:")
    print(df[col].dropna().unique().tolist())

## 6. Category Frequencies and Distribution

Frequency counts show how often each category occurs. Percentages make it easier to compare distributions across datasets of different sizes.

In [ ]:
for col in categorical_features:
    print(f"\n### {col}")
    counts = df[col].value_counts(dropna=False)
    percentages = df[col].value_counts(dropna=False, normalize=True).mul(100)
    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })
    display(summary)

    plot_counts = df[col].fillna("Missing").astype(str).value_counts()
    ax = plot_counts.plot(kind="bar", figsize=(8, 4), title=f"Category Distribution: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Detect Rare Categories

Rare categories occur infrequently. They can cause unstable estimates or create too many small groups. The threshold below is configurable; review the actual context before combining or removing categories.

In [ ]:
RARE_CATEGORY_MIN_COUNT = 2  # Adjust for your dataset

for col in categorical_features:
    counts = df[col].value_counts(dropna=False)
    rare = counts[counts < RARE_CATEGORY_MIN_COUNT]
    print(f"\n{col}: categories with fewer than {RARE_CATEGORY_MIN_COUNT} rows")
    if rare.empty:
        print("No rare categories found using this threshold.")
    else:
        display(rare.rename("Count").to_frame())

## 8. Category Imbalance

A categorical feature is unevenly distributed when a small number of categories dominate. This is not automatically an error, but it may affect comparisons and model behavior.

In [ ]:
for col in categorical_features:
    counts = df[col].value_counts(dropna=False)
    if len(counts) == 0:
        continue
    largest_share = counts.iloc[0] / counts.sum() * 100
    smallest_share = counts.iloc[-1] / counts.sum() * 100
    print(
        f"{col}: most common category = {counts.index[0]!r} "
        f"({largest_share:.1f}%), least common category = {counts.index[-1]!r} "
        f"({smallest_share:.1f}%)."
    )

## 9. Categorical Features vs Target

For classification, cross-tabulations show the target distribution within each category. Row-normalized percentages are often useful for comparing groups. These are descriptive associations and do not prove causation.

In [ ]:
if TARGET_COLUMN is not None:
    for col in categorical_features:
        print(f"\n### {col} vs {TARGET_COLUMN} — counts")
        display(pd.crosstab(df[col], df[TARGET_COLUMN], margins=True, dropna=False))

        print(f"### {col} vs {TARGET_COLUMN} — row percentages")
        row_percentages = pd.crosstab(
            df[col], df[TARGET_COLUMN], normalize="index", dropna=False
        ).mul(100).round(2)
        display(row_percentages)
else:
    print("No target configured. Skipping feature-versus-target analysis.")

In [ ]:
if TARGET_COLUMN is not None and PROBLEM_TYPE.lower() == "classification":
    target = df[TARGET_COLUMN]
    class_counts = target.value_counts(dropna=False)
    class_percentages = target.value_counts(dropna=False, normalize=True).mul(100)

    class_summary = pd.DataFrame({
        "Count": class_counts,
        "Percentage": class_percentages.round(2)
    })
    display(class_summary)

    if len(class_counts) > 1 and class_counts.sum() > 0:
        majority_share = class_counts.max() / class_counts.sum() * 100
        minority_share = class_counts.min() / class_counts.sum() * 100
        imbalance_ratio = class_counts.max() / class_counts.min() if class_counts.min() > 0 else np.inf
        print(f"Majority class share: {majority_share:.2f}%")
        print(f"Minority class share: {minority_share:.2f}%")
        print(f"Majority/minority count ratio: {imbalance_ratio:.2f}:1")
        if majority_share >= 80:
            print("Warning: one class represents at least 80% of observations; investigate possible imbalance.")
        else:
            print("No class reaches the illustrative 80% warning threshold. Still assess business impact.")

    ax = class_counts.plot(kind="bar", figsize=(7, 4), title=f"Class Distribution: {TARGET_COLUMN}")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

elif TARGET_COLUMN is not None:
    print("Classification analysis skipped because PROBLEM_TYPE is not 'classification'.")
else:
    print("No target configured.")

## 13. Regression Target Analysis

For regression, the target is numeric. Review center, spread, skewness, and potential outliers. Outliers should be investigated rather than automatically removed.

In [ ]:
if TARGET_COLUMN is not None and PROBLEM_TYPE.lower() == "regression":
    target_numeric = pd.to_numeric(df[TARGET_COLUMN], errors="coerce").dropna()

    if target_numeric.empty:
        print("The target has no usable numeric values. Check the target column and data types.")
    else:
        regression_summary = pd.Series({
            "Count": target_numeric.count(),
            "Mean": target_numeric.mean(),
            "Median": target_numeric.median(),
            "Variance (sample)": target_numeric.var(),
            "Standard Deviation (sample)": target_numeric.std(),
            "Skewness": target_numeric.skew(),
            "Minimum": target_numeric.min(),
            "Maximum": target_numeric.max()
        }, name="Value")
        display(regression_summary.to_frame())

        # IQR method for potential outliers
        q1 = target_numeric.quantile(0.25)
        q3 = target_numeric.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers = target_numeric[(target_numeric < lower_bound) | (target_numeric > upper_bound)]

        print(f"IQR lower bound: {lower_bound:.3f}")
        print(f"IQR upper bound: {upper_bound:.3f}")
        print(f"Potential outlier count: {len(outliers)}")
        display(outliers.to_frame(name=TARGET_COLUMN))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(target_numeric, bins=20, edgecolor="black")
        axes[0].set_title(f"Target Distribution: {TARGET_COLUMN}")
        axes[0].set_xlabel(TARGET_COLUMN)
        axes[0].set_ylabel("Frequency")

        axes[1].boxplot(target_numeric, vert=True)
        axes[1].set_title(f"Target Box Plot: {TARGET_COLUMN}")
        axes[1].set_ylabel(TARGET_COLUMN)

        plt.tight_layout()
        plt.show()

elif TARGET_COLUMN is not None:
    print("Regression analysis skipped because PROBLEM_TYPE is not 'regression'.")
else:
    print("No target configured.")